# Phase 0 — Data Archaeology (Downtime Prediction)

**Scope:** ONLY the 3 assets in `gold.dim_scope_asset`. Everything else is filtered out.

Goal: validate that the downtime-prediction problem is well-posed BEFORE we invest in
feature engineering or modeling. Specifically, answer per asset:

1. PI tag coverage timeline (distinct tags reporting per day, total values, gaps)
2. Running-indicator tag density and threshold sanity check
3. PI-derived downtime intervals from running-indicator with 5-min debounce
   (count, duration distribution, total hours)
4. GADS event landscape (events with valid REAL_START_DT in dim_date range,
   event types, durations)
5. Cross-check: do GADS events line up with PI-derived downtime intervals
   within +/- 2 hours? Confusion matrix.
6. HIGH-relevance tag sample density per asset per day (enough for 15-min grid?)

**Verdict cells at the bottom** summarize whether each asset is ready for
Phase 1 (label construction) and what blockers exist.

In [ ]:
from pyspark.sql import functions as F, Window as W

SCOPE = spark.table("gold.dim_scope_asset")
scope_ids = [r["asset_id"] for r in SCOPE.select("asset_id").collect()]
print("Scope assets:", scope_ids)
SCOPE.show(truncate=False)

## 1. PI tag coverage per scope asset

For each scope asset: distinct tags reporting, total PI value rows, earliest/latest
timestamp, daily distinct-tag density min/median/max.

In [ ]:
fact_pi_scope = (spark.table("gold.fact_pi")
    .filter(F.col("asset_id").isin(scope_ids)))

print("PI rows per scope asset (all match_sources)")
(fact_pi_scope.groupBy("asset_id","pi_match_source").count()
    .orderBy("asset_id","pi_match_source").show(truncate=False))

print("PI coverage span per asset")
(fact_pi_scope.groupBy("asset_id")
    .agg(F.countDistinct("Tag").alias("distinct_tags"),
         F.count("*").alias("rows"),
         F.min("Timestamp").alias("first_ts"),
         F.max("Timestamp").alias("last_ts"))
    .orderBy("asset_id").show(truncate=False))

In [ ]:
# Daily density: per (asset, date) -- distinct tags reporting that day
daily_density = (fact_pi_scope
    .withColumn("d", F.to_date("Timestamp"))
    .groupBy("asset_id","d")
    .agg(F.countDistinct("Tag").alias("tags_reporting"),
         F.count("*").alias("values"))
    .cache())

print("Daily tag-reporting density distribution per asset")
(daily_density.groupBy("asset_id")
    .agg(F.count("*").alias("days_with_data"),
         F.min("tags_reporting").alias("min_tags_per_day"),
         F.expr("percentile(tags_reporting, 0.5)").alias("p50_tags_per_day"),
         F.max("tags_reporting").alias("max_tags_per_day"),
         F.min("values").alias("min_values_per_day"),
         F.expr("percentile(values, 0.5)").alias("p50_values_per_day"),
         F.max("values").alias("max_values_per_day"))
    .orderBy("asset_id").show(truncate=False))

print("First 5 + last 5 days of data per asset")
for aid in scope_ids:
    print(f"\n--- {aid} ---")
    a = daily_density.filter(F.col("asset_id") == aid).orderBy("d")
    a.show(5, truncate=False)
    a.orderBy(F.col("d").desc()).show(5, truncate=False)

## 2. Running-indicator tag density + threshold sanity check

For each asset, the intake checklist defines one PI tag + threshold/operator that
means "asset is running". This is our **label source** for Phase 1.

Sanity check: does the running tag exist in fact_pi (under our asset_id), is it
numeric, and what's the distribution look like (so we can eyeball whether the
threshold is in the right ballpark)?

In [ ]:
run_ind = spark.table("gold.running_indicator")
print("running_indicator config (filtered to scope):")
run_ind.filter(F.col("asset_id").isin(scope_ids)).show(truncate=False)

# Join the running tag against fact_pi per asset
run_join = (run_ind
    .filter(F.col("asset_id").isin(scope_ids))
    .alias("ri")
    .join(spark.table("gold.fact_pi").alias("p"),
          (F.col("p.asset_id") == F.col("ri.asset_id")) &
          (F.col("p.Tag") == F.col("ri.Tag")), "left")
    .select(F.col("ri.asset_id").alias("asset_id"),
            F.col("ri.Tag").alias("Tag"),
            F.col("ri.operator").alias("operator"),
            F.col("ri.threshold_value").alias("threshold"),
            F.col("p.Timestamp").alias("Timestamp"),
            F.col("p.ValueNumeric").alias("ValueNumeric")))

print("\nrunning-tag coverage:")
(run_join.groupBy("asset_id","Tag","operator","threshold")
    .agg(F.count("ValueNumeric").alias("numeric_rows"),
         F.min("Timestamp").alias("first_ts"),
         F.max("Timestamp").alias("last_ts"),
         F.min("ValueNumeric").alias("v_min"),
         F.expr("percentile(ValueNumeric, 0.05)").alias("v_p05"),
         F.expr("percentile(ValueNumeric, 0.50)").alias("v_p50"),
         F.expr("percentile(ValueNumeric, 0.95)").alias("v_p95"),
         F.max("ValueNumeric").alias("v_max"))
    .orderBy("asset_id").show(truncate=False))

## 3. PI-derived downtime intervals

For each scope asset, evaluate the running predicate at every PI sample of the
running tag, then collapse consecutive same-state runs into intervals. Apply a
**5-minute debounce** so brief state flips don't fragment a real outage.

Output: per asset, count of downtime intervals + duration distribution +
total downtime hours over the full PI history.

In [ ]:
from pyspark.sql.types import BooleanType

# Apply running predicate. Operators we expect: '>', '>=', '<', '<=', '=='
def _running_expr(op, thr):
    op = (op or "").strip()
    if op == ">":   return F.col("ValueNumeric") >  F.lit(thr)
    if op == ">=":  return F.col("ValueNumeric") >= F.lit(thr)
    if op == "<":   return F.col("ValueNumeric") <  F.lit(thr)
    if op == "<=":  return F.col("ValueNumeric") <= F.lit(thr)
    if op == "==":  return F.col("ValueNumeric") == F.lit(thr)
    raise ValueError(f"unknown operator: {op!r}")

# Build a per-asset "running" boolean stream.
ri_rows = run_ind.filter(F.col("asset_id").isin(scope_ids)).collect()

import functools, operator as _op
running_dfs = []
for row in ri_rows:
    pred = _running_expr(row["operator"], row["threshold_value"])
    df = (spark.table("gold.fact_pi")
        .filter((F.col("asset_id") == row["asset_id"]) &
                (F.col("Tag") == row["Tag"]) &
                F.col("ValueNumeric").isNotNull())
        .select("asset_id","Timestamp",
                pred.alias("is_running")))
    running_dfs.append(df)

if running_dfs:
    running_all = functools.reduce(lambda a, b: a.unionByName(b), running_dfs).cache()
else:
    running_all = spark.createDataFrame([], "asset_id string, Timestamp timestamp, is_running boolean")

print("running-state row counts per asset:")
running_all.groupBy("asset_id","is_running").count().orderBy("asset_id","is_running").show()

In [ ]:
# Collapse consecutive same-state rows into intervals (per asset).
w = W.partitionBy("asset_id").orderBy("Timestamp")
state = (running_all
    .withColumn("prev_state", F.lag("is_running").over(w))
    .withColumn("changed", (F.col("prev_state").isNull()) |
                          (F.col("prev_state") != F.col("is_running")))
    .withColumn("grp", F.sum(F.col("changed").cast("int")).over(w)))

intervals = (state.groupBy("asset_id","grp","is_running")
    .agg(F.min("Timestamp").alias("start_ts"),
         F.max("Timestamp").alias("end_ts"),
         F.count("*").alias("samples"))
    .withColumn("duration_min",
        (F.col("end_ts").cast("long") - F.col("start_ts").cast("long")) / 60.0)
    .drop("grp"))

# Debounce: drop any DOWN interval shorter than 5 min, then re-collapse.
DEBOUNCE_MIN = 5
down_clean = (intervals
    .filter((F.col("is_running") == False) & (F.col("duration_min") >= DEBOUNCE_MIN))
    .select("asset_id","start_ts","end_ts","duration_min","samples")
    .cache())

print(f"down intervals per asset (>= {DEBOUNCE_MIN} min debounce):")
(down_clean.groupBy("asset_id")
    .agg(F.count("*").alias("down_events"),
         F.sum("duration_min").alias("total_down_min"),
         F.min("duration_min").alias("min_min"),
         F.expr("percentile(duration_min, 0.5)").alias("p50_min"),
         F.expr("percentile(duration_min, 0.9)").alias("p90_min"),
         F.max("duration_min").alias("max_min"))
    .withColumn("total_down_hours", F.round(F.col("total_down_min")/60.0, 1))
    .orderBy("asset_id").show(truncate=False))

# Persist for later phases.
(down_clean.write.mode("overwrite").format("delta")
    .option("overwriteSchema","true")
    .saveAsTable("gold.fact_pi_downtime_interval"))
print(f"gold.fact_pi_downtime_interval: {spark.table('gold.fact_pi_downtime_interval').count()} rows")

## 4. GADS event landscape (scope only)

Per scope asset: total events, events with valid REAL_START_DT in dim_date range,
event-type breakdown, duration distribution.

In [ ]:
fact_gads_scope = (spark.table("gold.fact_gads_event")
    .filter(F.col("asset_id").isin(scope_ids)))

print("GADS events per asset (by match source)")
(fact_gads_scope.groupBy("asset_id","gads_match_source").count()
    .orderBy("asset_id","gads_match_source").show(truncate=False))

print("Date-range coverage per asset")
(fact_gads_scope
    .withColumn("has_start_date_key", F.col("start_date_key").isNotNull())
    .groupBy("asset_id","has_start_date_key").count()
    .orderBy("asset_id","has_start_date_key").show(truncate=False))

# Top event types per asset
print("Top event types per scope asset:")
(fact_gads_scope.groupBy("asset_id","EVENT_TYPE_DESC").count()
    .orderBy("asset_id", F.col("count").desc()).show(40, truncate=False))

In [ ]:
# Duration distribution where both dates are valid
dur = (fact_gads_scope
    .filter(F.col("REAL_START_DT").isNotNull() & F.col("REAL_END_DT").isNotNull())
    .withColumn("event_hours",
        (F.col("REAL_END_DT").cast("long") - F.col("REAL_START_DT").cast("long")) / 3600.0)
    .filter(F.col("event_hours") > 0))

print("GADS event duration distribution per asset (events with valid start+end)")
(dur.groupBy("asset_id")
    .agg(F.count("*").alias("dated_events"),
         F.expr("percentile(event_hours, 0.1)").alias("p10_hrs"),
         F.expr("percentile(event_hours, 0.5)").alias("p50_hrs"),
         F.expr("percentile(event_hours, 0.9)").alias("p90_hrs"),
         F.max("event_hours").alias("max_hrs"),
         F.sum("event_hours").alias("total_event_hrs"))
    .orderBy("asset_id").show(truncate=False))

## 5. Cross-check: GADS event vs PI-derived downtime interval

For each GADS event with a valid REAL_START_DT, check whether there exists a
PI-derived downtime interval whose start is within +/- 2 hours of the GADS event
start (for the same asset).

This is our **trust calibration**: if GADS and PI agree, both sources are
plausible. If they disagree heavily, one source has structural issues we have to
solve before modeling.

In [ ]:
TOL_HOURS = 2
tol_sec = TOL_HOURS * 3600

gads_dated = (fact_gads_scope
    .filter(F.col("REAL_START_DT").isNotNull())
    .select("asset_id", "event_uid", "EVENT_TYPE_DESC",
            F.col("REAL_START_DT").alias("gads_start")))

pi_down = (spark.table("gold.fact_pi_downtime_interval")
    .select(F.col("asset_id").alias("p_asset"),
            F.col("start_ts").alias("p_start"),
            F.col("end_ts").alias("p_end"),
            F.col("duration_min").alias("p_dur_min")))

# match if a PI interval start is within tolerance of GADS start
matched = (gads_dated.alias("g")
    .join(pi_down.alias("p"),
          (F.col("g.asset_id") == F.col("p.p_asset")) &
          (F.abs(F.col("g.gads_start").cast("long") - F.col("p.p_start").cast("long")) <= tol_sec),
          "left")
    .withColumn("has_pi_match", F.col("p.p_start").isNotNull()))

# de-dup per gads event (closest PI interval if multiple)
w = W.partitionBy("event_uid").orderBy(
    F.abs(F.col("gads_start").cast("long") - F.col("p_start").cast("long")))
matched_one = (matched
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1).drop("rn"))

print(f"GADS-with-date events: matched within +/-{TOL_HOURS}h to a PI downtime interval")
(matched_one.groupBy("asset_id","has_pi_match").count()
    .orderBy("asset_id","has_pi_match").show(truncate=False))

print("\nReverse direction: PI downtime intervals NOT explained by any GADS event")
unmatched_pi = (pi_down.alias("p")
    .join(gads_dated.alias("g"),
          (F.col("g.asset_id") == F.col("p.p_asset")) &
          (F.abs(F.col("g.gads_start").cast("long") - F.col("p.p_start").cast("long")) <= tol_sec),
          "left_anti"))
(unmatched_pi.groupBy("p_asset").count()
    .withColumnRenamed("count","pi_intervals_without_gads")
    .orderBy("p_asset").show(truncate=False))

## 6. HIGH-relevance tag density (per asset, per day)

For Phase 2 we want a 15-min feature grid per asset. That means each HIGH/MED tag
needs ~96 samples/day on average to fill the grid without heavy imputation.

Per scope asset + HIGH-relevance tag: median samples per day. Flag tags with
median < 96/day (sparser than 15-min cadence).

In [ ]:
hi_density = (spark.table("gold.fact_pi")
    .filter(F.col("asset_id").isin(scope_ids))
    .filter(F.col("downtime_relevance") == "HIGH")
    .filter(F.col("ValueNumeric").isNotNull())
    .withColumn("d", F.to_date("Timestamp"))
    .groupBy("asset_id","Tag","d").count()
    .groupBy("asset_id","Tag")
    .agg(F.expr("percentile(count, 0.5)").alias("p50_samples_per_day"),
         F.count("*").alias("days_with_data"))
    .cache())

print("HIGH-relevance tag density per asset")
(hi_density.groupBy("asset_id")
    .agg(F.count("*").alias("hi_tags"),
         F.sum((F.col("p50_samples_per_day") < 96).cast("int")).alias("sparse_for_15min"),
         F.sum((F.col("p50_samples_per_day") >= 96).cast("int")).alias("ok_for_15min"),
         F.expr("percentile(p50_samples_per_day, 0.5)").alias("median_p50_samples"))
    .orderBy("asset_id").show(truncate=False))

print("Bottom 10 HIGH-relevance tags by median samples per day (per asset):")
for aid in scope_ids:
    print(f"\n--- {aid} ---")
    (hi_density.filter(F.col("asset_id") == aid)
        .orderBy("p50_samples_per_day").show(10, truncate=False))

## 7. Verdict — per asset, ready for Phase 1?

Pulls together the previous diagnostics into a single readable summary.

In [ ]:
from pyspark.sql import Row

verdict_rows = []
for aid in scope_ids:
    pi_total = fact_pi_scope.filter(F.col("asset_id") == aid).count()
    pi_first = (fact_pi_scope.filter(F.col("asset_id") == aid)
                .agg(F.min("Timestamp")).first()[0])
    pi_last  = (fact_pi_scope.filter(F.col("asset_id") == aid)
                .agg(F.max("Timestamp")).first()[0])
    has_ri = run_ind.filter(F.col("asset_id") == aid).count() > 0
    pi_down_n = (spark.table("gold.fact_pi_downtime_interval")
                 .filter(F.col("asset_id") == aid).count())
    gads_dated_n = (fact_gads_scope.filter(F.col("asset_id") == aid)
                    .filter(F.col("REAL_START_DT").isNotNull()).count())
    gads_total_n = fact_gads_scope.filter(F.col("asset_id") == aid).count()
    matched_n = (matched_one
                 .filter((F.col("asset_id") == aid) & (F.col("has_pi_match")))
                 .count())
    verdict_rows.append(Row(
        asset_id=aid,
        pi_rows=pi_total,
        pi_first=str(pi_first), pi_last=str(pi_last),
        has_running_indicator=has_ri,
        pi_derived_down_events=pi_down_n,
        gads_total=gads_total_n,
        gads_with_date=gads_dated_n,
        gads_matched_to_pi=matched_n,
        gads_match_pct=round(100.0 * matched_n / gads_dated_n, 1) if gads_dated_n else None,
    ))

verdict = spark.createDataFrame(verdict_rows)
print("=" * 72)
print("PHASE 0 VERDICT")
print("=" * 72)
verdict.show(truncate=False, vertical=True)

print("\nReadiness checklist:")
print("  [PASS] if pi_rows > 1M and has_running_indicator and pi_derived_down_events > 5")
print("  [WARN] if gads_match_pct < 30 -- PI and GADS disagree on outage timing")
print("  [FAIL] if no running indicator or no PI numeric data")